<style>
body {
    font-size: 20pt !important;
}

.rendered_html {
    font-size: 20pt !important;
}

.CodeMirror pre {
    font-size: 20pt !important;
}

.output pre {
    font-size: 20pt !important;
}
</style>


# Classification

Export a FITS file with ID, r, type, and class.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Data

In [ ]:
import numpy as np
from astropy.table import Table
from astropy.io import ascii
from scipy.spatial import Delaunay
import pandas as pd
from itertools import combinations

In [ ]:
def compute_r(df):
    coords = df[['x', 'y', 'z']].values
    types = df['type'].values

    tri = Delaunay(coords)

    neighbors = {i: set() for i in range(len(coords))}

    for simplex in tri.simplices:
        for i, j in combinations(simplex, 2):
            neighbors[i].add(j)
            neighbors[j].add(i)

    r = np.zeros(len(coords), dtype=float)

    for i, nbrs in neighbors.items():
        nbrs = list(nbrs)
        type_neighbors = types[nbrs]  # extraer los tipos de los vecinos
        n_data = np.sum(type_neighbors == 'data')
        n_rand = len(nbrs) - n_data

        if (n_data + n_rand) > 0:
            r[i] = (n_data - n_rand) / (n_data + n_rand)
        else:
            raise ValueError(f'No neighbors for point {i} in the triangulation.')

    out = df.copy()
    out['r'] = r
    return out

In [ ]:
def classification(data,number_rand):

    data.loc[(data['r'] >= -1.0) & (data['r'] <= -0.9), f'class_{number_rand}'] = 'void'
    data.loc[(data['r'] >  -0.9) & (data['r'] <=  0.0), f'class_{number_rand}'] = 'sheet'
    data.loc[(data['r'] >   0.0) & (data['r'] <=  0.9), f'class_{number_rand}'] = 'filament'
    data.loc[(data['r'] >   0.9) & (data['r'] <=  1.0), f'class_{number_rand}'] = 'knot'

    data.sort_values('z', inplace=True)

    return data

In [ ]:
filt_n1 = Table.read("/content/drive/MyDrive/DESI/data/LRG_NGC_data_filt1.ecsv").to_pandas()
filt_n2 = Table.read("/content/drive/MyDrive/DESI/data/LRG_NGC_data_filt2.ecsv").to_pandas()

In [ ]:
data = {'1':filt_n1,'2':filt_n2}

n_random = 100

for key, tbl in data.items():

    tbl = tbl.copy()
    tbl['type'] = 'data'

    for j in range(n_random):
        print(f'random {j}')
        df_rand = Table.read(f'/content/drive/MyDrive/DESI/rand/LRG_NGC_{key}_random_{j}_filt.fits').to_pandas()
        df_rand['type'] = 'rand'

        df_concat = pd.concat([df_rand, tbl], ignore_index=True)

        data_with_r = compute_r(df_concat)
        data_with_class = classification(data_with_r, j)

        final_data = data_with_class[['TARGETID', 'type', 'r', f'class_{j}']]

        final_data['type'] = final_data['type'].astype(str)
        final_data[f'class_{j}'] = final_data[f'class_{j}'].astype(str)

        table = Table.from_pandas(final_data)

        filename = f"/content/drive/MyDrive/DESI/classification/LRG_NGC_{key}_random_{j}_filt.fits"
        table.write(filename, format='fits', overwrite=True)


random 97


/tmp/ipython-input-6-1811954749.py:23: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  final_data['type'] = final_data['type'].astype(str)
/tmp/ipython-input-6-1811954749.py:24: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  final_data[f'class_{j}'] = final_data[f'class_{j}'].astype(str)


random 98


/tmp/ipython-input-6-1811954749.py:23: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  final_data['type'] = final_data['type'].astype(str)
/tmp/ipython-input-6-1811954749.py:24: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  final_data[f'class_{j}'] = final_data[f'class_{j}'].astype(str)


random 99


/tmp/ipython-input-6-1811954749.py:23: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  final_data['type'] = final_data['type'].astype(str)
/tmp/ipython-input-6-1811954749.py:24: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  final_data[f'class_{j}'] = final_data[f'class_{j}'].astype(str)
